In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

train_tfidf = load_npz("train_tfidf.npz")
valid_tfidf = load_npz("valid_tfidf.npz")
test_tfidf  = load_npz("test_tfidf.npz")

train_df = pd.read_csv("train_features.csv")
valid_df = pd.read_csv("valid_features.csv")
test_df  = pd.read_csv("test_features.csv")

y_train = train_df["label"].astype(int).values
y_valid = valid_df["label"].astype(int).values
y_test  = test_df["label"].astype(int).values

numeric_cols = train_df.columns.difference(["text", "tokens", "pos_seq", "label"])
X_train_extra = train_df[numeric_cols].astype(np.float64).values
X_valid_extra = valid_df[numeric_cols].astype(np.float64).values
X_test_extra  = test_df[numeric_cols].astype(np.float64).values

svd = TruncatedSVD(n_components=200, random_state=42)
X_train_svd = svd.fit_transform(train_tfidf)
X_valid_svd = svd.transform(valid_tfidf)
X_test_svd  = svd.transform(test_tfidf)

X_train = np.hstack([X_train_svd, X_train_extra])
X_valid = np.hstack([X_valid_svd, X_valid_extra])
X_test  = np.hstack([X_test_svd, X_test_extra])

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test  = scaler.transform(X_test)

print(f"Training shape: {X_train.shape}")
print(f"Validation shape: {X_valid.shape}")
print(f"Test shape: {X_test.shape}")

hidden_layer_sizes = [(128, 64), (256, 128), (128, 64, 32)]
alphas = [0.0001, 0.001]
learning_rates = ['constant', 'adaptive']

best_acc = -1
best_params = None
best_model = None

for hidden_layers in hidden_layer_sizes:
    for alpha in alphas:
        for lr in learning_rates:
            
            print(f"\nTrying: hidden_layers={hidden_layers}, alpha={alpha}, learning_rate={lr}")
            
            model = MLPClassifier(
                hidden_layer_sizes=hidden_layers,
                activation='relu',
                solver='adam',
                alpha=alpha,
                learning_rate=lr,
                max_iter=100,
                early_stopping=True,
                validation_fraction=0.1,
                random_state=42,
                verbose=False
            )
            
            model.fit(X_train, y_train)
            
            preds_valid = model.predict(X_valid)
            acc_valid = accuracy_score(y_valid, preds_valid)
            
            print(f" → Validation accuracy: {acc_valid:.4f}")
            
            if acc_valid > best_acc:
                best_acc = acc_valid
                best_params = (hidden_layers, alpha, lr)
                best_model = model

print("\n" + "="*50)
print("BEST NEURAL NETWORK MODEL")
print("="*50)
print("Hidden layers:", best_params[0])
print("Alpha (L2 penalty):", best_params[1])
print("Learning rate:", best_params[2])
print("Validation Accuracy:", best_acc)

test_preds = best_model.predict(X_test)
test_acc = accuracy_score(y_test, test_preds)

print("\nFINAL TEST ACCURACY:", test_acc)
print("\nClassification Report:")
print(classification_report(y_test, test_preds))

Training shape: (21464, 259)
Validation shape: (716, 259)
Test shape: (966, 259)

Trying: hidden_layers=(128, 64), alpha=0.0001, learning_rate=constant
 → Validation accuracy: 0.8087

Trying: hidden_layers=(128, 64), alpha=0.0001, learning_rate=adaptive
 → Validation accuracy: 0.8087

Trying: hidden_layers=(128, 64), alpha=0.001, learning_rate=constant
 → Validation accuracy: 0.8031

Trying: hidden_layers=(128, 64), alpha=0.001, learning_rate=adaptive
 → Validation accuracy: 0.8031

Trying: hidden_layers=(256, 128), alpha=0.0001, learning_rate=constant
 → Validation accuracy: 0.8282

Trying: hidden_layers=(256, 128), alpha=0.0001, learning_rate=adaptive
 → Validation accuracy: 0.8282

Trying: hidden_layers=(256, 128), alpha=0.001, learning_rate=constant
 → Validation accuracy: 0.8254

Trying: hidden_layers=(256, 128), alpha=0.001, learning_rate=adaptive
 → Validation accuracy: 0.8254

Trying: hidden_layers=(128, 64, 32), alpha=0.0001, learning_rate=constant
 → Validation accuracy: 0.82